In [1]:
!pip install conllu
!pip install unsloth==2026.4.2

from google.colab import drive
drive.mount('/content/drive')
"""
#!unzip /content/drive/MyDrive/parseme.zip -d /content/drive/MyDrive/parseme
#!mkdir -p /content/drive/MyDrive/parseme_data

!for f in /content/drive/MyDrive/parseme/*.tgz; do \
  tar -xzf "$f" -C /content/drive/MyDrive/parseme_data; \
done
"""
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
import gc
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

from transformers import set_seed


import pandas as pd
import conllu
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainingArguments
import pandas as pd
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig
###
#code adapted from Google Search's Gemini @ google.com



def file_to_parse(file_path):
    with open(file_path, 'r', encoding = 'utf-8') as f:
        data = f.read()

    sentences = conllu.parse(data)
    return sentences
###



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_11490/2179873115.py:27: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import re
def find_compounds(sentences, add_to_vmwe = False, categorized = False):

  def recursive_compound_labeling(sentence, index, label, value):

    compound = sentence[index]['head']-1


    if 'compound' not in sentence[index]['deprel']:
      if 'COMP' not in sentence[index]['parseme:mwe']:
        if sentence[index]['parseme:mwe'] == '*':
          sentence[index]['parseme:mwe'] = str(value) + label
        else:
          #something is wrong
          pass
      return sentence, [index]



    elif 'compound' in sentence[index]['deprel'] and sentence[compound]['parseme:mwe'] != '*':
      value = sentence[compound]['parseme:mwe'].split(':')[0]

      if sentence[index]['parseme:mwe'] == '*':
        sentence[index]['parseme:mwe'] = str(value)
      else:
        pass
        #somethings wrong
      return sentence, [index, compound]

    else:
      sentence = recursive_compound_labeling(sentence,compound,label,value)
      return recursive_compound_labeling(sentence[0], index,label, value)[0], sentence[1] + [index]



  if add_to_vmwe == True:
    pass
  else:
    #can't be in more than one compound using PARSEME
    for i in range(len(sentences)):
      skip = []
      count = 1
      for j in range(len(sentences[i])):
        sentences[i][j]['parseme:mwe'] = '*'
      for j in range(len(sentences[i])):

        if 'COMP' not in sentences[i][j]['parseme:mwe']:
          if 'compound' in sentences[i][j]['deprel']:

              #recursive compound detection
            if j in skip:
              pass
            else:
              label = ':COMP'
              #recursive labeling
              #print(i)
              if categorized == True:
                typ = sentences[i][j]['deprel'].split(':')
                if len(typ) > 1:
                  label = ':COMP-' + typ[1].upper()
              e = recursive_compound_labeling(sentences[i],j,label,count)

              sentences[i] = e[0]
              #print(len(e[0]))
              skip = skip + e[1]
              count = count+1

  return sentences

def parse_to_dict(sentences):
  inputs = []
  labels = []
  for i in sentences:
    inputt = []
    label = []
    for j in range(len(i)):
        inputt.append(i[j]['form'])
        label.append(i[j]['parseme:mwe'])
    inputs.append(inputt)
    labels.append(label)
  dictionary = {'sentence': inputs, 'label': labels}
  return dictionary


In [3]:
def find_spans(labels, usage = 'full'):
  #assumes labels are like 1:VID, 1, etc
  #usage = 'full' means it outputs everything in between in a non consecutive span
  #this method works for one sentence inputs
  spans = {}
  if usage == 'full':
    for i in range(len(labels)):
      if labels[i] != '*':
        split = labels[i].split(';')
        for j in split:
          try:
            int(j)
            ### for compounds only
            if str(j) not in spans:
              spans[str(j)] = [i]
            else:
              if isinstance(spans[str(j)][len(spans[str(j)])-1],str):
                if len(spans[str(j)]) == 2:
                  spans[str(j)].insert(1,i+1)
                else:
                  spans[str(j)][1] = i+1
                #spans[str(j)][0:1] = spans[str(j)][0:1].sort()
              else:
                if len(spans[str(j)]) == 1:
                  spans[str(j)].append(i+1)
                else:
                  spans[str(j)][1] = i+1
                #spans[str(j)][0:1] = spans[str(j)][0:1].sort()

          except ValueError:
            a = j.split(':')
            if a[0] in spans:
              if len(spans[a[0]]) == 1:
                spans[a[0]].append(i+1)
              else:
                spans[a[0]][1] = i+1
              #spans[a[0]][0:1] = spans[a[0]][0:1].sort()
              spans[a[0]].append(a[1])
            else:
              spans[a[0]] = [i, a[1]]
  else:
    for i in range(len(labels)):
      if labels[i] != '*':
        split = labels[i].split(';')
        for j in split:
          try:
            int(j)
            if str(j) not in spans:
              spans[str(j)] = [i]
            else:
              if isinstance(spans[str(j)][len(spans[str(j)])-1],str):
                spans[str(j)].insert(len(spans[str(j)])-1,i)
                #spans[str(j)][0:len(spans[str(j)])-1] = spans[str(j)][0:len(spans[str(j)])-1].sort()
              else:
                spans[str(j)].append(i)
                #spans[str(j)] = spans[str(j)].sort()
          except ValueError:
            a = j.split(':')
            if a[0] not in spans:
              spans[a[0]] = [i, a[1]]
            else:
              spans[a[0]].append(i)
              spans[a[0]].append(a[1])

  if usage == 'nominal_only':
    for m in spans:
      if spans[m][len(spans[m])-1] == 'COMP':
        if len(spans[m]) >= 3:
          aa = sorted(spans[m][0:len(spans[m])-1])
          new_list = []
          for n in range(aa[0],aa[len(aa)-1]+1):
            new_list.append(n)
          new_list.append(spans[m][len(spans[m])-1])
          spans[m] = new_list

  #print(spans)
  return spans



pr = """¿Puedes identificar y etiquetar todas las VMWE (expresiones verbales multi-palabra) en esta oración?

Los tipos de VMWE incluyen, entre otros, modismos verbales, construcciones verbales ligeras, verbos inherentemente reflexivos, construcciones verbo-partícula, construcciones con múltiples verbos y verbos inherentemente adposicionales.

Ejemplos de VMWE por categoría incluyen:

VID: estaba a favor, quedar bien
MVC: podría utilizar
LVC.full: tiene consecuencias, hace referencia
IAV: insistió en
IRV: se comprometían, se separaron

Tu etiqueta debe consistir en el tipo de VMWE, seguido de dos puntos y luego la VMWE. Si hay varias VMWE, etiqueta cada una en una línea separada. Si no hay ninguna VMWE, escribe 'No MWE found.'

"""


def sentence_to_text(sentences, prompt = pr, usage = 'full'):
  #usage = full for full spans, anything else for only the labeled MWE
  prompts = {'prompt': [], 'sentence': [], 'output': []}
  a = parse_to_dict(sentences)

  for i in range(len(a['sentence'])):
    response = ''
    sentence = ' '.join(a['sentence'][i])
    prompts['prompt'].append(prompt)
    prompts['sentence'].append(sentence)
    #print(i)
    spans = find_spans(a['label'][i], usage)
    if usage=='full':
      for j in spans:
        #print(spans[j])
        b = a['sentence'][i][spans[j][0]:spans[j][1]]
        b = ' '.join(b)
        response = response + spans[j][2] + ': ' + b + '\n'
    else:
      for j in spans:
        b = [a['sentence'][i][k] for k in spans[j][0:len(spans[j])-1]]
        b = ' '.join(b)
        response = response + spans[j][len(spans[j])-1] + ': ' + b + '\n'
    response = response[0:len(response)-1] #get rid of the last newline
    if response == '':
      response = 'No MWE found.'
    #print(response)
    prompts['output'].append(response)
  return prompts



In [4]:

###code from chatgpt
def formatt(instance):
  return {
      "messages": [
          {"role": "user", "content": instance['prompt'] + "\n\nEsta es la oración:\n" + instance['sentence']},
          {"role": "assistant", "content": instance['output']}
      ]
  }
"""
def template(instance, tokenizer, tk = False):
  return{
      "text": tokenizer.apply_chat_template(
          instance["messages"],
          tokenize = tk
      )
  }
"""
###

'\ndef template(instance, tokenizer, tk = False):\n  return{\n      "text": tokenizer.apply_chat_template(\n          instance["messages"],\n          tokenize = tk\n      )\n  }\n'

In [5]:


from datasets import concatenate_datasets
import random
import numpy as np
#print(tokenizer.chat_template)
path = '/content/drive/MyDrive/capstone/'
file_path = path+'parseme_data/ES/train.cupt'

rebalances = True



def rebalance(unbalanced_data, seed):
  ##adapted from gpt code
  positive = unbalanced_data.filter(lambda x: x['output'] != 'No MWE found.')
  rng = np.random.default_rng(seed)
  select_pos = rng.choice(len(positive), size = 1075, replace = True)
  positive = positive.select(select_pos.tolist())

  negative = unbalanced_data.filter(lambda x: x['output'] == 'No MWE found.')
  rng = np.random.default_rng(seed)
  select_neg = rng.choice(len(negative), size = 1075, replace = True)
  negative = negative.select(select_neg.tolist())

  dataset = concatenate_datasets([positive, negative])
  dataset = dataset.shuffle(seed=seed)

  return dataset

  ###


for i in range(0,5):
  set_seed(i)
  sentences = file_to_parse(file_path)

  e = 0
  while e < len(sentences):
    if len(sentences[e]) > 125:
      sentences.pop(e)
      e = e-1
    e = e+1
  """
  if len(sentences)>2150:
    random.seed(42)
    sentences = random.sample(sentences, k = 2150)
    #print(sentences[0]) #ensure its the same every time
  """
  data = sentence_to_text(sentences, usage = 'not_full')

  data = Dataset.from_dict(data)

  if rebalances == True:
    data = rebalance(data,i)

  dataset = data.map(formatt)
  dataset.remove_columns(dataset.column_names)


  qwen = 'Qwen/Qwen3-32B'
  ###code from chatgpt
  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = qwen,
      max_seq_length = 2048,
      dtype = torch.bfloat16,
      load_in_4bit = False
  )

  model = FastLanguageModel.get_peft_model(
      model,
      r = 16,
      target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
      lora_alpha = 16,
      lora_dropout = 0,
      bias = 'none'
  )
  ###
  if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token
  model.config.pad_token_id = tokenizer.pad_token_id


  #data = data.map(template, fn_kwargs={'tokenizer': tokenizer})


  ###gpt code
  def template(instance):
    if isinstance(instance['messages'][0],dict):
        return [tokenizer.apply_chat_template(
              instance['messages'],
              tokenize = False,
              add_generation_prompt = False
        )]
    else:
      return [
          tokenizer.apply_chat_template(
              messages,
              tokenize = False,
              add_generation_prompt = False
          )
        for messages in instance['messages']
      ]
    ###

  training_args = SFTConfig(num_train_epochs = 3,
                                    logging_strategy = 'steps',
                                    logging_steps = 1000,
                                    save_strategy = 'no',
                                    report_to = 'none',
                                    learning_rate = 2e-5,
                                    packing = False,
                                    assistant_only_loss = True,
                                    max_seq_length = 2048,
                                    dataset_text_field = None
                            )
  trainer = SFTTrainer(model = model,
                      tokenizer = tokenizer,
                      train_dataset = dataset,
                      args = training_args,
                      formatting_func = template)

  ###
  trainer.train()
  model.save_pretrained(path + '/models/qwen-32b/vmwe_fewshot_es_'+str(i))

  trainer.model = None
  trainer.optimizer = None
  trainer.lr_scheduler = None

  del trainer
  del model
  del data
  del dataset

  gc.collect()
  torch.cuda.empty_cache()
  torch.cuda.ipc_collect()

Filter:   0%|          | 0/3424 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3424 [00:00<?, ? examples/s]

Map:   0%|          | 0/2150 [00:00<?, ? examples/s]

==((====))==  Unsloth 2026.4.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

...

unsloth/Qwen3-32B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["None"] (num_proc=52):   0%|          | 0/2150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,150 | Num Epochs = 3 | Total steps = 807
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 134,217,728 of 32,896,340,992 (0.41% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


In [6]:
path = '/content/drive/MyDrive/capstone/'
file_path = path+'parseme_data/AR/train.cupt'
a = file_to_parse(path+'parseme_data/AR/train.cupt')
e = 0
while e < len(a):
  if len(a[e]) > 125:
    print('del')
    a.pop(e)
    e = e-1
  e = e+1


#a = find_compounds(a, False, True)
a = sentence_to_text(a, usage = 'not_full')



del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del
del


In [7]:
data = rebalance(data)
data['sentence'][0]

NameError: name 'data' is not defined